# C7-cnn-transfer — Practice p27 — Solution

**Type:** scenario analysis · **Difficulty:** advanced · **Concepts:** layer-freezing, requires-grad, cnn-training

## Part I — committed independent-control predictions

In [ ]:
prediction_eval: tuple[bool, bool, bool, bool] = (True, True, False, False)
prediction_frozen: tuple[bool, bool, bool, bool] = (False, False, True, False)
prediction_inference: tuple[bool, bool, bool, bool] = (False, False, True, True)

committed_predictions = [prediction_eval, prediction_frozen, prediction_inference]  # PLAN017_MUTATION_TARGET: C7-p27-committed-predictions

Evaluation mode changes BatchNorm's normalization rule and stops its running-buffer updates, but it does not disable autograd. Freezing every parameter removes this output's graph because the input also does not require gradients, while training-mode BatchNorm still updates its buffers and produces the baseline value. Inference mode independently removes graph construction and makes an inference tensor; because the trunk stays in training mode, its numerical output matches the baseline and its BatchNorm buffers still move. Thus mode, parameter trainability, and graph context control different pieces of state.

### MARKED VERIFICATION CELL

This cell fails closed and prints only one agreement bit; it never reveals per-case observations.

In [ ]:
# PLAN017_VERIFIER: C7-p27-committed-predictions
from copy import deepcopy

import torch
from torch import nn

if len(committed_predictions) != 3 or any(
    value is Ellipsis
    or not isinstance(value, tuple)
    or len(value) != 4
    or any(type(flag) is not bool for flag in value)
    for value in committed_predictions
):
    raise RuntimeError("commit exactly three four-bool prediction tuples first")

SEED = 20260804
torch.manual_seed(SEED)
base_trunk = nn.Sequential(
    nn.Conv2d(3, 5, kernel_size=3, padding=1, bias=False),
    nn.BatchNorm2d(5),
    nn.ReLU(),
)
x = torch.randn(4, 3, 7, 7)

def run_case(*, training, parameters_require_grad, inference):
    trunk = deepcopy(base_trunk)
    trunk.train(training)
    for parameter in trunk.parameters():
        parameter.requires_grad_(parameters_require_grad)
    bn = trunk[1]
    before_mean = bn.running_mean.clone()
    before_var = bn.running_var.clone()
    context = torch.inference_mode() if inference else torch.enable_grad()
    with context:
        output = trunk(x)
    buffers_change = not (
        torch.allclose(bn.running_mean, before_mean, atol=1e-7, rtol=0)
        and torch.allclose(bn.running_var, before_var, atol=1e-7, rtol=0)
    )
    return (
        bool(output.requires_grad), output.detach().clone(), buffers_change,
        bool(torch.is_inference(output)),
    )

_, baseline_output, _, _ = run_case(
    training=True, parameters_require_grad=True, inference=False
)
settings = [
    {"training": False, "parameters_require_grad": True, "inference": False},
    {"training": True, "parameters_require_grad": False, "inference": False},
    {"training": True, "parameters_require_grad": True, "inference": True},
]
observed = []
for case in settings:
    graph, output, buffers, is_inference = run_case(**case)
    changed = not torch.allclose(output, baseline_output, atol=1e-7, rtol=0)
    observed.append((graph, changed, buffers, is_inference))
all_predictions_agree = all(
    hand == seen for hand, seen in zip(committed_predictions, observed)
)
print("all_agree:", all_predictions_agree)
del observed, baseline_output

## Part II — selective training and evaluation audit

In [ ]:
if all_predictions_agree is not True:
    raise RuntimeError("Part I must be committed and correct before Part II")

torch.set_default_dtype(torch.float64)
torch.manual_seed(SEED)

class TrainingAuditCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 5, 3, padding=1)
        self.bn = nn.BatchNorm2d(5)
        self.dropout = nn.Dropout(p=0.15)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Linear(5, 3)

    def forward(self, value):
        value = torch.relu(self.bn(self.conv(value)))
        value = self.dropout(value)
        return self.head(torch.flatten(self.pool(value), 1))

template = TrainingAuditCNN()
generator = torch.Generator(device="cpu").manual_seed(SEED)
train_X = 0.04 * torch.randn(18, 1, 7, 7, generator=generator)
train_y = torch.arange(18, dtype=torch.long) % 3
train_X[train_y == 0, :, :, 1:3] += 1.0
train_X[train_y == 1, :, 4:6, :] += 1.0
diagonal = torch.arange(7)
class_two = train_X[train_y == 2].clone()
class_two[:, :, diagonal, diagonal] += 1.0
train_X[train_y == 2] = class_two
train_X[train_y == 0] -= 0.8
train_X[train_y == 2] += 0.8
criterion = nn.CrossEntropyLoss()

In [ ]:
train_model = deepcopy(template)
for name, parameter in train_model.named_parameters():
    parameter.requires_grad_(not name.startswith(("conv", "bn")))

parameter_before = {
    name: parameter.detach().clone() for name, parameter in train_model.named_parameters()
}
buffer_before = {
    name: buffer.detach().clone() for name, buffer in train_model.named_buffers()
}
trainable_parameters = [
    parameter for parameter in train_model.parameters() if parameter.requires_grad
]
optimizer = torch.optim.SGD(trainable_parameters, lr=0.18)
loss_history = []
train_model.train()
for _ in range(18):
    optimizer.zero_grad(set_to_none=True)
    logits = train_model(train_X)
    loss = criterion(logits, train_y)
    loss_history.append(float(loss.detach()))
    loss.backward()
    optimizer.step()

expected_frozen_names = sorted(
    name for name, _ in train_model.named_parameters() if name.startswith(("conv", "bn"))
)
expected_trainable_names = sorted(
    name for name, _ in train_model.named_parameters() if name not in expected_frozen_names
)
optimizer_parameter_objects = [
    parameter for group in optimizer.param_groups for parameter in group["params"]
]
optimizer_parameter_ids = {id(parameter) for parameter in optimizer_parameter_objects}
train_mode_audit = train_model.training and all(
    module.training for module in train_model.modules()
)
trainability_audit = (
    sorted(name for name, parameter in train_model.named_parameters() if not parameter.requires_grad)
    == expected_frozen_names
    and sorted(name for name, parameter in train_model.named_parameters() if parameter.requires_grad)
    == expected_trainable_names
)
optimizer_ownership_audit = (
    optimizer_parameter_ids
    == {
        id(parameter)
        for name, parameter in train_model.named_parameters()
        if name in expected_trainable_names
    }
    and len(optimizer_parameter_objects) == len(expected_trainable_names)
)
gradient_names = sorted(
    name for name, parameter in train_model.named_parameters() if parameter.grad is not None
)
frozen_parameters_unchanged = all(
    torch.equal(parameter.detach(), parameter_before[name])
    for name, parameter in train_model.named_parameters()
    if name in expected_frozen_names
)
moved_parameter_names = sorted(
    name
    for name, parameter in train_model.named_parameters()
    if not torch.equal(parameter.detach(), parameter_before[name])
)
allowed_parameter_movement = bool(
    moved_parameter_names
    and set(moved_parameter_names).issubset(expected_trainable_names)
)
training_buffers_moved = any(
    name in {"bn.running_mean", "bn.running_var"}
    and not torch.equal(buffer.detach(), buffer_before[name])
    for name, buffer in train_model.named_buffers()
)
loss_audit = bool(
    len(loss_history) == 18
    and torch.isfinite(torch.tensor(loss_history)).all()
    and loss_history[-1] <= 0.80 * loss_history[0]
)

trained_parameter_state = {
    name: parameter.detach().clone() for name, parameter in train_model.named_parameters()
}
trained_buffer_state = {
    name: buffer.detach().clone() for name, buffer in train_model.named_buffers()
}
train_model.eval()  # PLAN017_MUTATION_TARGET: C7-p27-eval-mode
with torch.no_grad():
    evaluation_logits_a = train_model(train_X)
    evaluation_logits_b = train_model(train_X)

evaluation_mode_audit = (
    not train_model.training and all(not module.training for module in train_model.modules())
)
evaluation_has_no_graph = (
    not evaluation_logits_a.requires_grad and not evaluation_logits_b.requires_grad
)
evaluation_logits_repeat = torch.allclose(
    evaluation_logits_a, evaluation_logits_b, atol=1e-10, rtol=1e-8
)
evaluation_parameters_unchanged = all(
    torch.equal(parameter.detach(), trained_parameter_state[name])
    for name, parameter in train_model.named_parameters()
)
evaluation_buffers_unchanged = all(
    torch.equal(buffer.detach(), trained_buffer_state[name])
    for name, buffer in train_model.named_buffers()
)

training_evaluation_certificate = bool(
    all_predictions_agree
    and train_mode_audit
    and trainability_audit
    and optimizer_ownership_audit
    and gradient_names == expected_trainable_names
    and frozen_parameters_unchanged
    and allowed_parameter_movement
    and training_buffers_moved
    and loss_audit
    and evaluation_mode_audit
    and evaluation_has_no_graph
    and evaluation_logits_repeat
    and evaluation_parameters_unchanged
    and evaluation_buffers_unchanged
)

### Answer check

In [ ]:
# PLAN017_ANSWER_CHECK: C7-p27-mode-buffer-audit
assert committed_predictions == [
    (True, True, False, False),
    (False, False, True, False),
    (False, False, True, True),
]
assert all_predictions_agree is True
assert expected_frozen_names == ["bn.bias", "bn.weight", "conv.bias", "conv.weight"]
assert expected_trainable_names == ["head.bias", "head.weight"]
assert train_mode_audit
assert trainability_audit
assert optimizer_parameter_ids == {
    id(parameter)
    for name, parameter in train_model.named_parameters()
    if name in expected_trainable_names
}
assert len(optimizer_parameter_objects) == len(expected_trainable_names)
assert optimizer_ownership_audit
assert gradient_names == expected_trainable_names
assert all(
    torch.equal(parameter.detach(), parameter_before[name])
    for name, parameter in train_model.named_parameters()
    if name in expected_frozen_names
)
assert frozen_parameters_unchanged
assert moved_parameter_names
assert set(moved_parameter_names).issubset(expected_trainable_names)
assert allowed_parameter_movement
assert any(
    not torch.equal(trained_buffer_state[name], buffer_before[name])
    for name in ("bn.running_mean", "bn.running_var")
)
assert training_buffers_moved
assert len(loss_history) == 18
assert torch.isfinite(torch.tensor(loss_history)).all()
assert loss_history[-1] <= 0.80 * loss_history[0] and loss_audit
assert not train_model.training
assert all(not module.training for module in train_model.modules())
assert evaluation_mode_audit
assert not evaluation_logits_a.requires_grad and not evaluation_logits_b.requires_grad
assert evaluation_has_no_graph
assert torch.allclose(evaluation_logits_a, evaluation_logits_b, atol=1e-10, rtol=1e-8)
assert evaluation_logits_repeat
assert all(
    torch.equal(parameter.detach(), trained_parameter_state[name])
    for name, parameter in train_model.named_parameters()
)
assert evaluation_parameters_unchanged
assert all(
    torch.equal(buffer.detach(), trained_buffer_state[name])
    for name, buffer in train_model.named_buffers()
)
assert evaluation_buffers_unchanged
assert training_evaluation_certificate